# 語音 LoRA：Colab 練習

先閱讀第一課並取得錄音授權。切換至 GPU 執行階段；此筆記本會下載模型與套件，並讀寫私人 Google Drive。請逐格執行與確認輸出。

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import torch
assert torch.cuda.is_available(), "請在執行階段設定選擇 GPU"
print("GPU:", torch.cuda.get_device_name(0))


## 1. 取得教材與依賴

環境版本可能隨 Colab 更新而變；安裝後如提示重新啟動，請重新執行前面儲存格。

In [ ]:
!git clone https://github.com/RichTsai1688/tts-lora-learning.git /content/tts-lora-learning
%cd /content/tts-lora-learning
%pip install -q qwen-tts==0.1.1 peft==0.18.1 tensorboard


## 2. 設定私有資料與實驗名稱

CSV 中的音訊路徑以 CSV 所在資料夾為基準；參考錄音須為另一段同說話者錄音。每次重跑請更換 RUN_NAME。

In [ ]:
from pathlib import Path
import subprocess, sys
DATA = Path("/content/drive/MyDrive/tts-private")  # 修改此處
MANIFEST = DATA / "manifest.csv"
REFERENCE = DATA / "reference.wav"
RUN_NAME = "lesson-smoke-01"  # 每次執行使用新名稱
RUN = DATA / "runs" / RUN_NAME
SMOKE = True  # 小量測試成功後設為 False，並改 RUN_NAME
PRECISION = "fp16"  # 支援 bf16 的 GPU 可改為 bf16
assert MANIFEST.is_file() and REFERENCE.is_file(), "請確認私有資料位置"
assert not RUN.exists(), "此實驗名稱已存在，請修改 RUN_NAME"
RUN.mkdir(parents=True)


## 3. 盤點與分組

閱讀報表，人工檢查授權、音訊／文字吻合、各組語言與語調分布；不符合時修正資料並更換實驗名稱。

In [ ]:
subprocess.run([sys.executable, "scripts/dataset_tool.py", str(MANIFEST), "--ref-audio", str(REFERENCE), "--output", str(RUN / "raw"), "--seed", "42"], check=True)


## 4. 將錄音轉為模型訓練格式

會下載 tokenizer，依資料量花費時間。

In [ ]:
encoded = RUN / "encoded"
encoded.mkdir()
for split in ("train", "val", "test"):
    subprocess.run([sys.executable, "training/prepare_data.py", "--device", "cuda:0", "--input_jsonl", str(RUN / "raw" / f"{split}_raw.jsonl"), "--output_jsonl", str(encoded / f"{split}.jsonl")], check=True)


## 5. 訓練與檢查

先做 smoke test；正式訓練請重新設定實驗名稱與 SMOKE。下載基礎模型與輸出合併模型需要足夠空間；fp16 模式以 float32 載入基礎權重，可能需要較多 GPU 記憶體。

In [ ]:
cmd = [sys.executable, "training/train_lora.py", "--init_model_path", "Qwen/Qwen3-TTS-12Hz-0.6B-Base", "--dataset_dir", str(encoded), "--output_model_path", str(RUN / "model"), "--speaker_name", "speaker_demo", "--batch_size", "1", "--num_epochs", "1" if SMOKE else "3", "--precision", PRECISION]
if SMOKE:
    cmd += ["--max_train_batches", "4", "--max_eval_batches", "1"]
subprocess.run(cmd, check=True)
print("私人輸出位置:", RUN / "model")


檢視 `loss_history.csv` 與 `final_model/`。這是合併後的完整模型，請用未參與訓練的固定測試句人工聽測並記錄限制。